In [7]:
from typing import Any, Dict, Optional

import numpy as np
from scipy.optimize import minimize_scalar


def loss(
    w: np.ndarray,
    ret: np.ndarray,
    risk_aversion: float = 2.0,
) -> float:
    """目的関数"""
    return float(np.exp(-risk_aversion * (ret @ w)).sum())


def grad(
    w: np.ndarray,
    ret: np.ndarray,
    risk_aversion: float = 2.0,
) -> np.ndarray:
    """目的関数の勾配"""
    return -risk_aversion * (
        np.exp(-risk_aversion * (ret @ w)).reshape(-1, 1) * ret
    ).sum(axis=0)


def line_search_obj(
    gamma: float,
    w_curr: np.ndarray,
    w_prev: np.ndarray,
    ret: np.ndarray,
    risk_aversion: float,
) -> float:
    """ラインサーチの目的関数"""
    return loss(
        (1.0 - gamma) * w_prev + gamma * w_curr,
        ret=ret,
        risk_aversion=risk_aversion,
    )


def line_search(
    w_curr: np.ndarray,
    w_prev: np.ndarray,
    ret: np.ndarray,
    risk_aversion: float,
) -> float:
    """ラインサーチ"""
    result = minimize_scalar(
        line_search_obj, bounds=(0.0, 1.0), args=(w_curr, w_prev, ret, risk_aversion)
    )
    return result.x


def fw_exponential(
    ret: np.ndarray,
    w0: Optional[np.ndarray] = None,
    max_iter: int = 1000,
    risk_aversion: float = 2.0,
    use_line_search: bool = False,
) -> Dict[str, Any]:
    """Frank-Wolfeアルゴリズムによって指数効用関数を最適化する。

    Args:
        ret: 日次リターンの配列 (T * J)
        w0: 初期値 (J,)
        max_iter: 反復回数
        risk_aversion: リスク回避度 (alpha)
        use_line_search: Frank-WolfeアルゴリズムのStep2でラインサーチをする
    """
    J = ret.shape[1]

    if w0 is None:
        w0 = np.ones(J, dtype=ret.dtype) / J

    history = {
        "w": [],
        "loss": [],
        "grad": [],
    }
    w = w0
    history["w"].append(w.copy())
    history["loss"].append(loss(w=w, ret=ret, risk_aversion=risk_aversion))
    for i in range(max_iter):
        # 勾配を計算する。
        g = grad(w, ret, risk_aversion)

        # Step 1: 確率単体上の線形最適化
        # ここでは、勾配の成分のargminを計算するだけ！
        idx = np.argmin(g, keepdims=True)[0]

        s = np.zeros(J, dtype=ret.dtype)
        s[idx] = 1.0


        # Step 2: 固定ステップ幅での更新、またはラインサーチ
        if not use_line_search:
            gamma = 2.0 / (2.0 + i)
        else:
            gamma = line_search(s, w, ret, risk_aversion)
        w = (1.0 - gamma) * w + gamma * s
        
        history["grad"].append(g.copy())
        history["w"].append(w.copy())
        history["loss"].append(loss(w, ret, risk_aversion))
    return history